# Preparation occurrence of ecosystem types at res. 0.25°

In [ ]:
# Libraries
import os
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_data =  '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'

---

### Pre-processing

In [ ]:
def prep_cop_r001():
    # Get prepared copernicus luc data
    ds = xr.open_zarr(f'{dir01}ds_prep_cop_r001.zarr')
    
    # Prepare and export nc files for regridding to r025
    for i in ds.data_vars:
    
        # Get array
        da = ds[i]
        
        # Add lat and lon attributes (necessary for regridding)
        ds.lat.attrs = dict(standard_name = 'latitude',
                            long_name = 'latitude',
                            units = 'degrees_north',
                            axis = 'Y')
            
        ds.lon.attrs = dict(standard_name = 'longitude',
                            long_name = 'longitude',
                            units = 'degrees_east',
                            axis = 'X')
    
        # Export
        da.to_netcdf(f'{dir07}ds_prep_{i}_r001.nc', mode='w') 


prep_cop_r001()

---

### Regridding of COP LUC to 0.25° res.

In [ ]:
%%bash 
cd ../paper_ecosystem_carbon/output/07_eval_prep/

module load cdo

for i in \
    bare builtup crops grass moss_lichen permanent_water seasonal_water \
    shrub snow tree forest_unknown forest_enf forest_ebf forest_dnf \
    forest_dbf forest_mixed
do
    cdo -P 24 remapycon,../data/luh2_v2h/states.nc \
    ds_prep_cop_${i}_r001.nc ds_prep_cop_${i}_rcon_r025.nc

done

In [ ]:
# Merge regridded files
ds = xr.open_mfdataset(f'{dir07}ds_prep_cop_*_rcon_r025.nc')

# Export dataset as zarr
ds.to_zarr(f'{dir07}ds_prep_cop_rcon_r025.zarr', mode='w');

# Remove intermediate data
for etype in list(ds.data_vars):
    os.remove(dir07 + f'ds_prep_{etype}_r001.nc')
    os.remove(dir07 + f'ds_prep_{etype}_rcon_r025.nc')

---

In [ ]:
# Get regridded copernicus and prepared dinerstein etype data
ds_cop = xr.open_zarr(f'{dir07}ds_prep_cop_rcon_r025.zarr')
ds_din = xr.open_zarr(f'{dir01}ds_prep_dinerstein_etype_r025.zarr')

In [ ]:
# Create and export dataset with occurance of etypes
def prep_occur(etype):

    #trop_ef
    if etype == 'trop_ef':
        da_din_sel = ds_din.trop_ef
        da_cop_sel = xr.where((ds_cop.cop_forest_ebf > 0) | 
                              (ds_cop.cop_forest_enf > 0), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 0, True, False)
    
    #trop_df
    if etype == 'trop_df':
        da_din_sel = ds_din.trop_df
        da_cop_sel = xr.where((ds_cop.cop_forest_dbf > 0) | 
                              (ds_cop.cop_forest_dnf > 0), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 0, True, False)
    
    #temp_ebf
    if etype == 'temp_ebf':
        da_din_sel = ds_din.temp_ebf
        da_cop_sel = xr.where(ds_cop.cop_forest_ebf > 0, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 0, True, False)
    
    #temp_dbf
    if etype == 'temp_dbf':
        da_din_sel = ds_din.temp_dbf
        da_cop_sel = xr.where(ds_cop.cop_forest_dbf > 0, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 0, True, False)
    
    #temp_enf
    if etype == 'temp_enf':
        da_din_sel = ds_din.temp_enf
        da_cop_sel = xr.where(ds_cop.cop_forest_enf > 0, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 0, True, False)
    
    #temp_dnf
    if etype == 'temp_dnf':
        da_din_sel = ds_din.temp_dnf
        da_cop_sel = xr.where(ds_cop.cop_forest_dnf > 0, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 0, True, False)
    
    #glob_shrub
    if etype == 'glob_shrub':
        da_din_sel = ds_din.glob_shrub
        da_cop_sel = xr.where(ds_cop.cop_shrub > 0, True, False)
        da_cop_sel2 = da_cop_sel
    
    #glob_grass
    if etype == 'glob_grass':
        da_din_sel = ds_din.glob_grass
        da_cop_sel = xr.where(ds_cop.cop_grass > 0, True, False)
        da_cop_sel2 = da_cop_sel
    
    #glob_tundr
    if etype == 'glob_tundr':
        da_din_sel = ds_din.glob_tundr
        da_cop_sel = da_din_sel
        da_cop_sel2 = da_din_sel
    
    #glob_mount
    if etype == 'glob_mount':
        da_din_sel = ds_din.glob_mount
        da_cop_sel = da_din_sel
        da_cop_sel2 = da_din_sel
    
    # glob crop
    if etype == 'glob_crop':
        da_din_sel = xr.ones_like(ds_din.trop_ef).astype('bool')
        da_cop_sel = xr.where((ds_cop.cop_crops > 0), True, False)
        da_cop_sel2 = da_cop_sel
    
    # glob_past
    if etype == 'glob_past':
        da_din_sel = xr.ones_like(ds_din.trop_ef).astype('bool')
        da_cop_sel = xr.where((ds_cop.cop_grass > 0), True, False)
        da_cop_sel = da_cop_sel
    
    
    da_occur = xr.where((da_din_sel == True) & (da_cop_sel == True) & 
                        (da_cop_sel2 == True), True, False)
    
    return da_occur


ds_occur = xr.Dataset()
for etype in ds_din.data_vars:
    ds_occur[etype] = prep_occur(etype)


ds_occur.to_zarr(f'{dir07}ds_prep_occur_r025.zarr', mode='w');

In [ ]:
# Create and export dataset with occurance of etypes
def prep_occur_dom(etype):

    #trop_ef
    if etype == 'trop_ef':
        da_din_sel = ds_din.trop_ef
        da_cop_sel = xr.where((ds_cop.cop_forest_ebf > 50) | 
                              (ds_cop.cop_forest_enf > 50), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 50, True, False)
    
    #trop_df
    if etype == 'trop_df':
        da_din_sel = ds_din.trop_df
        da_cop_sel = xr.where((ds_cop.cop_forest_dbf > 50) | 
                              (ds_cop.cop_forest_dnf > 50), True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 50, True, False)
    
    #temp_ebf
    if etype == 'temp_ebf':
        da_din_sel = ds_din.temp_ebf
        da_cop_sel = xr.where(ds_cop.cop_forest_ebf > 50, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 50, True, False)
    
    #temp_dbf
    if etype == 'temp_dbf':
        da_din_sel = ds_din.temp_dbf
        da_cop_sel = xr.where(ds_cop.cop_forest_dbf > 50, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 50, True, False)
    
    #temp_enf
    if etype == 'temp_enf':
        da_din_sel = ds_din.temp_enf
        da_cop_sel = xr.where(ds_cop.cop_forest_enf > 50, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 50, True, False)
    
    #temp_dnf
    if etype == 'temp_dnf':
        da_din_sel = ds_din.temp_dnf
        da_cop_sel = xr.where(ds_cop.cop_forest_dnf > 50, True, False)
        da_cop_sel2 = xr.where(ds_cop.cop_tree > 50, True, False)
    
    #glob_shrub
    if etype == 'glob_shrub':
        da_din_sel = ds_din.glob_shrub
        da_cop_sel = xr.where(ds_cop.cop_shrub > 50, True, False)
        da_cop_sel2 = da_cop_sel
    
    #glob_grass
    if etype == 'glob_grass':
        da_din_sel = ds_din.glob_grass
        da_cop_sel = xr.where(ds_cop.cop_grass > 50, True, False)
        da_cop_sel2 = da_cop_sel
    
    #glob_tundr
    if etype == 'glob_tundr':
        da_din_sel = ds_din.glob_tundr
        da_cop_sel = da_din_sel
        da_cop_sel2 = da_din_sel
    
    #glob_mount
    if etype == 'glob_mount':
        da_din_sel = ds_din.glob_mount
        da_cop_sel = da_din_sel
        da_cop_sel2 = da_din_sel
    
    # glob crop
    if etype == 'glob_crop':
        da_din_sel = xr.ones_like(ds_din.trop_ef).astype('bool')
        da_cop_sel = xr.where((ds_cop.cop_crops > 50), True, False)
        da_cop_sel2 = da_cop_sel
    
    # glob_past
    if etype == 'glob_past':
        da_din_sel = xr.ones_like(ds_din.trop_ef).astype('bool')
        da_cop_sel = xr.where((ds_cop.cop_grass > 50), True, False)
        da_cop_sel = da_cop_sel
    
    
    da_occur = xr.where((da_din_sel == True) & (da_cop_sel == True) & 
                        (da_cop_sel2 == True), True, False)
    
    return da_occur


ds_occur_dom = xr.Dataset()
for etype in ds_din.data_vars:
    ds_occur_dom[etype] = prep_occur_dom(etype)


ds_occur_dom.to_zarr(f'{dir07}ds_prep_occur_dom_r025.zarr', mode='w');